In [ ]:
#!pip install facenet-pytorch

In [ ]:
#!pip install selenium opencv-python requests pillow

In [ ]:
import time
import os
import urllib.request
import cv2
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By

# Initialize the face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Define minimum resolution threshold (e.g., 100x100 pixels)
MIN_WIDTH = 100
MIN_HEIGHT = 100

# Resize target dimensions
RESIZE_WIDTH = 128
RESIZE_HEIGHT = 128

def is_face_present(image_path):
    """Check if the image contains a face using OpenCV."""
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) > 0:
        return True
    return False

def is_low_quality(image_path):
    """Check if the image has low resolution."""
    img = cv2.imread(image_path)
    height, width = img.shape[:2]

    # If the width or height is below the threshold, consider the image low quality
    if width < MIN_WIDTH or height < MIN_HEIGHT:
        return True
    return False

def resize_image(image_path, target_size=(RESIZE_WIDTH, RESIZE_HEIGHT)):
    """Resize the image to the target size."""
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, target_size)
    return img_resized

def scrape_google_images_selenium(query, save_dir, num_images=50):
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=options)

    search_url = f"https://www.google.com/search?q={query}&tbm=isch"
    driver.get(search_url)
    time.sleep(5)

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    images = driver.find_elements(By.TAG_NAME, 'img')
    count = 0
    for img in images:
        src = img.get_attribute('src')
        if src and count < num_images:
            try:
                # Create a temporary file to save the image
                file_name = f"{save_dir}/img_{count}.jpg"
                urllib.request.urlretrieve(src, file_name)

                # Check if the image is low quality
                if is_low_quality(file_name):
                    print(f"Low-quality image (resolution too low), deleting image: {file_name}")
                    os.remove(file_name)  # Delete low-quality image
                    continue

                # Check if the image contains a face
                if is_face_present(file_name):
                    # Resize the image before saving
                    resized_image = resize_image(file_name)

                    # Save resized image
                    resized_file_name = f"{save_dir}/resized_img_{count}.jpg"
                    cv2.imwrite(resized_file_name, resized_image)
                    print(f"Face detected, saving resized image: {resized_file_name}")

                    # Remove the original image after saving the resized one
                    os.remove(file_name)
                    count += 1
                else:
                    print(f"No face detected, deleting image: {file_name}")
                    os.remove(file_name)  # Delete irrelevant image

            except Exception as e:
                print(f"Error downloading image {src}: {e}")

    driver.quit()
    print(f"Downloaded {count} relevant, high-quality, resized images to {save_dir}")

# Example usage
scrape_google_images_selenium("top hollywood film actor headshot", 'filmactor', num_images=50)
scrape_google_images_selenium("top hollywood film child actor headshot", 'littlefilmactor', num_images=50)


In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image

# Directory containing the images
image_dir = 'filmactor'

# List images
images = os.listdir(image_dir)

# Display a few images
for i, img_name in enumerate(images[:10]):  # Change the number to view more
    img_path = os.path.join(image_dir, img_name)
    img = Image.open(img_path)
    plt.figure()
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Image {i+1}")
    plt.show()


In [42]:
import shutil

# Replace 'your_folder_name' with the name of your folder
folder_path = 'sample_data'

# Delete the folder and all its contents
shutil.rmtree(folder_path)
print(f"Folder {folder_path} and its contents have been deleted.")

Folder sample_data and its contents have been deleted.
